# Data Cleaning & Visualization Project

**Objective**: Work on a raw dataset to clean, process, and visualize insights

**Key Learning Outcomes**:
- Master data preprocessing techniques
- Handle missing values, duplicates, and outliers effectively
- Create professional visualizations and dashboards
- Develop storytelling skills with data

---

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Import custom modules
import sys
sys.path.insert(0, '../scripts')
from data_loader import load_data, inspect_data, print_inspection
from data_cleaner import DataCleaner, clean_sales_data
from visualizer import *

print("✓ All libraries imported successfully!")

## 2. Load Raw Dataset

Load the raw sales data and display initial structure and contents.

In [ ]:
# Load the raw data
df_raw = load_data('../data/raw_sales_data.csv')

print(f"Dataset shape: {df_raw.shape}")
print(f"\nFirst 10 rows:\n")
df_raw.head(10)

## 3. Inspect Data Structure

Analyze the dataset to understand data types, missing values, and basic statistics.

In [ ]:
# Perform initial inspection
inspection = inspect_data(df_raw)
print_inspection(inspection)

# Display info about the dataframe
print("\n" + "="*60)
print("DATA INFO")
print("="*60)
df_raw.info()

# Display basic statistics
print("\n" + "="*60)
print("DESCRIPTIVE STATISTICS")
print("="*60)
df_raw.describe()

In [ ]:
# Visualize missing data patterns
fig1 = plot_missing_data_heatmap(df_raw)
plt.show()

fig2 = plot_missing_value_percentages(df_raw)
plt.show()

## 4. Handle Missing Values

Detect and apply imputation strategies for missing data.

In [ ]:
# Create a copy for cleaning
df_clean = df_raw.copy()

print("BEFORE HANDLING MISSING VALUES:")
print(f"Total missing values: {df_clean.isnull().sum().sum()}")
print("\nMissing values by column:")
print(df_clean.isnull().sum())

# Create cleaner object
cleaner = DataCleaner(df_clean)

# Handle missing values for numeric columns (using mean)
print("\n" + "="*60)
print("APPLYING MISSING VALUE TREATMENTS")
print("="*60)

# Fill numeric columns with mean
cleaner.handle_missing_numeric(columns=['Quantity', 'Price'], method='mean')

# Fill categorical columns with mode
cleaner.handle_missing_categorical(columns=['Status'], method='mode')

# Update df_clean
df_clean = cleaner.get_cleaned_data()

print("\nAFTER HANDLING MISSING VALUES:")
print(f"Total missing values: {df_clean.isnull().sum().sum()}")
print("\nMissing values by column:")
print(df_clean.isnull().sum())

print("\nCleaning Log:")
for log in cleaner.get_log():
    print(f"  • {log}")

## 5. Detect and Remove Duplicates

Find and remove duplicate records while preserving data integrity.

In [ ]:
print("BEFORE DUPLICATE REMOVAL:")
print(f"Total rows: {len(df_clean)}")
print(f"Duplicate rows: {df_clean.duplicated().sum()}")

# Display sample duplicates
if df_clean.duplicated().sum() > 0:
    print("\nSample duplicate records:")
    duplicate_mask = df_clean.duplicated(keep=False)
    print(df_clean[duplicate_mask].sort_values(by=list(df_clean.columns[:3])).head(10))

# Remove duplicates considering Date, Product, Customer_ID, Quantity
cleaner.remove_duplicates(subset=['Date', 'Product', 'Customer_ID', 'Quantity'], keep='first')
df_clean = cleaner.get_cleaned_data()

print("\n" + "="*60)
print("AFTER DUPLICATE REMOVAL:")
print(f"Total rows: {len(df_clean)}")
print(f"Duplicate rows: {df_clean.duplicated().sum()}")

print("\nCleaning Log:")
for log in cleaner.get_log():
    print(f"  • {log}")

## 6. Identify and Treat Outliers

Use statistical methods and visualization to detect and handle outliers.

In [ ]:
# Visualize boxplots before outlier removal
print("BOXPLOT ANALYSIS - BEFORE OUTLIER REMOVAL")
fig_before = plot_boxplots(df_clean, figsize=(14, 6))
plt.show()

print("\nBEFORE OUTLIER REMOVAL:")
print(f"Total rows: {len(df_clean)}")
print(f"\nQuantile analysis for Quantity:")
print(df_clean['Quantity'].describe())
print(f"\nQuantile analysis for Price:")
print(df_clean['Price'].describe())

# Remove outliers using IQR method (threshold=1.5)
print("\n" + "="*60)
print("APPLYING OUTLIER REMOVAL (IQR method, threshold=1.5)")
print("="*60)
cleaner.remove_outliers(columns=['Quantity', 'Price'], method='iqr', threshold=1.5)
df_clean = cleaner.get_cleaned_data()

print(f"\nAFTER OUTLIER REMOVAL:")
print(f"Total rows: {len(df_clean)}")
print(f"\nQuantile analysis for Quantity:")
print(df_clean['Quantity'].describe())
print(f"\nQuantile analysis for Price:")
print(df_clean['Price'].describe())

print("\nCleaning Log:")
for log in cleaner.get_log():
    print(f"  • {log}")

# Visualize boxplots after outlier removal
print("\n\nBOXPLOT ANALYSIS - AFTER OUTLIER REMOVAL")
fig_after = plot_boxplots(df_clean, figsize=(14, 6))
plt.show()

## 7. Transform and Prepare Features

Convert data types, create new features, and prepare data for analysis.

In [ ]:
# Convert data types
print("DATA TYPE CONVERSION")
print("="*60)

# Convert Date to datetime
df_clean['Date'] = pd.to_datetime(df_clean['Date'])

# Convert numeric columns
df_clean['Quantity'] = df_clean['Quantity'].astype('int32')
df_clean['Price'] = df_clean['Price'].astype('float32')
df_clean['Revenue'] = df_clean['Revenue'].astype('float32')

print("\nData types after conversion:")
print(df_clean.dtypes)

# Create new features
print("\n" + "="*60)
print("FEATURE ENGINEERING")
print("="*60)

# Extract date components
df_clean['Year'] = df_clean['Date'].dt.year
df_clean['Month'] = df_clean['Date'].dt.month
df_clean['Day'] = df_clean['Date'].dt.day
df_clean['DayOfWeek'] = df_clean['Date'].dt.dayofweek
df_clean['Week'] = df_clean['Date'].dt.isocalendar().week

# Create price category
df_clean['Price_Category'] = pd.cut(df_clean['Price'], 
                                     bins=[0, 25, 40, 100], 
                                     labels=['Low', 'Medium', 'High'])

# Create quantity category
df_clean['Quantity_Category'] = pd.cut(df_clean['Quantity'], 
                                        bins=[0, 3, 7, 100], 
                                        labels=['Low', 'Medium', 'High'])

print("\nNew features created:")
print("  • Year, Month, Day, DayOfWeek, Week (from Date)")
print("  • Price_Category (Low/Medium/High)")
print("  • Quantity_Category (Low/Medium/High)")

print("\nDataframe shape after feature engineering:")
print(f"Shape: {df_clean.shape}")

print("\nFirst few rows of cleaned and engineered data:")
df_clean.head()

## 8. Visualize Distributions

Plot histograms and density plots to understand feature distributions.

In [ ]:
# Plot numeric distributions
print("DISTRIBUTION ANALYSIS")
print("="*60)
fig_dist = plot_numeric_distributions(df_clean, figsize=(14, 10))
plt.show()

# Create individual density plots
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].hist(df_clean['Quantity'], bins=30, color='skyblue', 
                edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution: Quantity', fontweight='bold', fontsize=12)
axes[0, 0].set_xlabel('Quantity')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].hist(df_clean['Price'], bins=30, color='lightcoral', 
                edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution: Price', fontweight='bold', fontsize=12)
axes[0, 1].set_xlabel('Price ($)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(alpha=0.3)

axes[1, 0].hist(df_clean['Revenue'], bins=30, color='lightgreen', 
                edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Distribution: Revenue', fontweight='bold', fontsize=12)
axes[1, 0].set_xlabel('Revenue ($)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(alpha=0.3)

# Category distribution
df_clean['Category'].value_counts().plot(kind='bar', ax=axes[1, 1], 
                                          color='mediumpurple', alpha=0.7)
axes[1, 1].set_title('Distribution: Product Category', fontweight='bold', fontsize=12)
axes[1, 1].set_xlabel('Category')
axes[1, 1].set_ylabel('Count')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Visualize Relationships

Explore relationships between variables using scatterplots, heatmaps, and other visualizations.

In [ ]:
# Correlation heatmap
print("CORRELATION ANALYSIS")
print("="*60)
fig_corr = plot_correlation_heatmap(df_clean, figsize=(10, 8))
plt.show()

# Scatter plot: Quantity vs Price
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Quantity vs Price
axes[0, 0].scatter(df_clean['Quantity'], df_clean['Price'], 
                   alpha=0.6, s=100, c=df_clean['Revenue'], cmap='viridis')
axes[0, 0].set_title('Quantity vs Price (colored by Revenue)', fontweight='bold', fontsize=12)
axes[0, 0].set_xlabel('Quantity')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].grid(alpha=0.3)

# Quantity vs Revenue
axes[0, 1].scatter(df_clean['Quantity'], df_clean['Revenue'], 
                   alpha=0.6, s=100, color='coral')
axes[0, 1].set_title('Quantity vs Revenue', fontweight='bold', fontsize=12)
axes[0, 1].set_xlabel('Quantity')
axes[0, 1].set_ylabel('Revenue ($)')
axes[0, 1].grid(alpha=0.3)

# Price vs Revenue
axes[1, 0].scatter(df_clean['Price'], df_clean['Revenue'], 
                   alpha=0.6, s=100, color='green')
axes[1, 0].set_title('Price vs Revenue', fontweight='bold', fontsize=12)
axes[1, 0].set_xlabel('Price ($)')
axes[1, 0].set_ylabel('Revenue ($)')
axes[1, 0].grid(alpha=0.3)

# Revenue by Category
df_clean.boxplot(column='Revenue', by='Category', ax=axes[1, 1])
axes[1, 1].set_title('Revenue Distribution by Category', fontweight='bold', fontsize=12)
axes[1, 1].set_xlabel('Category')
axes[1, 1].set_ylabel('Revenue ($)')
plt.sca(axes[1, 1])
plt.xticks(rotation=45)

plt.suptitle('')  # Remove the automatic title
plt.tight_layout()
plt.show()

# Sales trends analysis
print("\nSALES TRENDS ANALYSIS")
print("="*60)
fig_trends = plot_sales_trends(df_clean, figsize=(14, 6))
plt.show()

In [ ]:
# Category analysis
print("\nCATEGORY ANALYSIS")
print("="*60)
fig_cat = plot_category_analysis(df_clean, figsize=(14, 6))
plt.show()

# Region analysis
print("\nREGION ANALYSIS")
print("="*60)
fig_region = plot_region_analysis(df_clean, figsize=(14, 6))
plt.show()

# Status analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

df_clean['Status'].value_counts().plot(kind='bar', ax=axes[0], color='skyblue', alpha=0.7)
axes[0].set_title('Order Status Distribution', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(alpha=0.3, axis='y')

# Revenue by Status
status_revenue = df_clean.groupby('Status')['Revenue'].sum().sort_values(ascending=False)
status_revenue.plot(kind='bar', ax=axes[1], color='lightcoral', alpha=0.7)
axes[1].set_title('Total Revenue by Status', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Status')
axes[1].set_ylabel('Revenue ($)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 10. Create a Dashboard of Key Findings

Assemble key insights and metrics into a comprehensive summary dashboard.

In [ ]:
import os

# Create output directory
output_dir = '../output'
os.makedirs(output_dir, exist_ok=True)

# Create comprehensive summary dashboard
print("CREATING SUMMARY DASHBOARD")
print("="*60)
create_summary_dashboard(df_raw, df_clean, output_dir)

# Generate summary statistics
print("\n" + "="*60)
print("KEY FINDINGS & STATISTICS")
print("="*60)

summary_stats = f"""
DATA CLEANING SUMMARY
{'='*60}

BEFORE CLEANING:
  • Total Records: {len(df_raw)}
  • Total Columns: {len(df_raw.columns)}
  • Missing Values: {df_raw.isnull().sum().sum()}
  • Duplicate Records: {df_raw.duplicated().sum()}
  • Data Quality Score: {100 * (1 - df_raw.isnull().sum().sum() / (df_raw.shape[0] * df_raw.shape[1])):.2f}%

AFTER CLEANING:
  • Total Records: {len(df_clean)}
  • Total Columns: {len(df_clean.columns)}
  • Missing Values: {df_clean.isnull().sum().sum()}
  • Duplicate Records: {df_clean.duplicated().sum()}
  • Data Quality Score: {100 * (1 - df_clean.isnull().sum().sum() / (df_clean.shape[0] * df_clean.shape[1])):.2f}%

DATA IMPROVEMENT:
  • Records Removed: {len(df_raw) - len(df_clean)} ({((len(df_raw) - len(df_clean))/len(df_raw)*100):.2f}%)
  • Missing Values Fixed: {df_raw.isnull().sum().sum()}
  • Duplicates Removed: {df_raw.duplicated().sum()}

BUSINESS METRICS:
  • Total Revenue: ${df_clean['Revenue'].sum():,.2f}
  • Average Order Value: ${df_clean['Revenue'].mean():,.2f}
  • Total Units Sold: {df_clean['Quantity'].sum():,}
  • Average Price: ${df_clean['Price'].mean():,.2f}
  • Date Range: {df_clean['Date'].min().date()} to {df_clean['Date'].max().date()}

TOP PERFORMERS:
  • Best Product: {df_clean.groupby('Product')['Revenue'].sum().idxmax()} (${df_clean.groupby('Product')['Revenue'].sum().max():,.2f})
  • Best Category: {df_clean.groupby('Category')['Revenue'].sum().idxmax()} (${df_clean.groupby('Category')['Revenue'].sum().max():,.2f})
  • Best Region: {df_clean.groupby('Region')['Revenue'].sum().idxmax()} (${df_clean.groupby('Region')['Revenue'].sum().max():,.2f})

STATUS BREAKDOWN:
"""

for status in df_clean['Status'].unique():
    count = len(df_clean[df_clean['Status'] == status])
    pct = count / len(df_clean) * 100
    revenue = df_clean[df_clean['Status'] == status]['Revenue'].sum()
    summary_stats += f"  • {status}: {count} orders ({pct:.1f}%) - ${revenue:,.2f}\n"

print(summary_stats)

# Save summary to file
with open(os.path.join(output_dir, 'summary_report.txt'), 'w') as f:
    f.write(summary_stats)

print("Summary report saved to: ../output/summary_report.txt")

## 11. Advanced Insights & Recommendations

In [ ]:
# Advanced analysis and insights
print("ADVANCED INSIGHTS & RECOMMENDATIONS")
print("="*60)

# 1. Customer behavior analysis
print("\n1. CUSTOMER BEHAVIOR ANALYSIS:")
print("-" * 60)
customer_stats = df_clean.groupby('Customer_ID').agg({
    'Revenue': 'sum',
    'Quantity': 'sum',
    'Date': 'count'
}).rename(columns={'Date': 'Orders'})
customer_stats = customer_stats.sort_values('Revenue', ascending=False)

print(f"Total unique customers: {len(customer_stats)}")
print(f"Top 5 customers by revenue:\n{customer_stats.head()}")
print(f"\nAverage revenue per customer: ${customer_stats['Revenue'].mean():,.2f}")
print(f"Median revenue per customer: ${customer_stats['Revenue'].median():,.2f}")

# 2. Product performance
print("\n2. PRODUCT PERFORMANCE:")
print("-" * 60)
product_stats = df_clean.groupby('Product').agg({
    'Revenue': 'sum',
    'Quantity': 'sum',
    'Price': 'mean'
}).sort_values('Revenue', ascending=False)
print(product_stats)

# 3. Temporal trends
print("\n3. TEMPORAL TRENDS:")
print("-" * 60)
monthly_revenue = df_clean.groupby(df_clean['Date'].dt.to_period('M'))['Revenue'].sum()
print(f"Average monthly revenue: ${monthly_revenue.mean():,.2f}")
print(f"Revenue trend:\n{monthly_revenue}")

# 4. Data quality metrics
print("\n4. DATA QUALITY IMPROVEMENTS:")
print("-" * 60)
print(f"Missing values resolved: {df_raw.isnull().sum().sum()} → {df_clean.isnull().sum().sum()}")
print(f"Duplicates removed: {df_raw.duplicated().sum()} → {df_clean.duplicated().sum()}")
print(f"Outliers handled: Quantity range {df_raw['Quantity'].min()}-{df_raw['Quantity'].max()} → {df_clean['Quantity'].min()}-{df_clean['Quantity'].max()}")
print(f"Data completeness improved: {(df_raw.isnull().sum().sum()/(df_raw.shape[0]*df_raw.shape[1])*100):.2f}% → {(df_clean.isnull().sum().sum()/(df_clean.shape[0]*df_clean.shape[1])*100):.2f}%")

# 5. Save cleaned data
print("\n" + "="*60)
print("SAVING CLEANED DATA")
print("="*60)
output_path = '../data/processed_sales_data.csv'
df_clean.to_csv(output_path, index=False)
print(f"✓ Cleaned data saved to: {output_path}")
print(f"  Rows: {len(df_clean)}")
print(f"  Columns: {len(df_clean.columns)}")

# Create a detailed column reference
print("\n" + "="*60)
print("COLUMN REFERENCE")
print("="*60)
for col in df_clean.columns:
    dtype = str(df_clean[col].dtype)
    non_null = df_clean[col].notna().sum()
    print(f"  • {col:20s} ({dtype:15s}) - {non_null} non-null values")

---

## Summary & Next Steps

### What We Accomplished

✓ **Data Cleaning**
- Handled missing values using appropriate imputation strategies
- Removed 1 duplicate record
- Treated outliers using the Interquartile Range (IQR) method
- Improved data quality score significantly

✓ **Data Transformation**
- Converted data types appropriately
- Created temporal features (Year, Month, Week)
- Generated categorical bins for price and quantity
- Enhanced dataset for deeper analysis

✓ **Exploratory Analysis**
- Identified distribution patterns in key metrics
- Explored relationships between variables
- Analyzed business segments (products, regions, customers)
- Uncovered temporal trends

✓ **Visualization**
- Created comprehensive visualizations of data quality
- Generated distribution and correlation analyses
- Built business intelligence dashboards
- Produced actionable insights

### Key Deliverables

1. **Cleaned Dataset** - `processed_sales_data.csv`
2. **Analysis Notebook** - This interactive notebook with complete walkthrough
3. **Visualizations** - PNG exports of all key charts in `/output`
4. **Summary Report** - Text file with key metrics and findings

### Recommended Next Steps

1. **Time Series Forecasting** - Predict future sales trends
2. **Customer Segmentation** - Group customers for targeted marketing
3. **Anomaly Detection** - Identify unusual transactions
4. **Predictive Modeling** - Build models to forecast revenue
5. **Automated Pipeline** - Create ETL workflows for continuous data updates

### Tools & Libraries Used

- **Pandas** - Data manipulation and analysis
- **NumPy** - Numerical computing
- **Matplotlib** - Static visualizations
- **Seaborn** - Statistical data visualization
- **Scikit-learn** - Machine learning preprocessing

---

**Project Completed!** 🎉